## Reproducing ResNet on CIFAR-10: Experiments on Network Depth, Batch Size, and Pooling

---
- Baseline Code Link: https://github.com/kuangliu/pytorch-cifar

In [1]:
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from torchsummary import summary

import os
import argparse

import matplotlib.pyplot as plt
import numpy as np

# from resnet_20_32_44_56_v1 import *
from resnet_20_32_44_56_v2 import *
from utils import progress_bar

In [2]:
parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
parser.add_argument('--resume', '-r', action='store_true',
                    help='resume from checkpoint')
# args = parser.parse_args()
args, _ = parser.parse_known_args()

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")
best_acc = 0   # best test accuracy
start_epoch = 0   # start from epoch 0 or last checkpoint epoch

device: mps


In [3]:
# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

==> Preparing data..
Files already downloaded and verified
Files already downloaded and verified


In [5]:
# Model
print('==> Building Model..\n')

net = ResNet20()
# net = ResNet32()
# net = ResNet44()
# net = ResNet56()

# Check layer(type), output shape, param #
print('==> Model Summary')
summary(net, (3, 32, 32))

net = net.to(device)

if args.resume:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

criterion = nn.CrossEntropyLoss()
# 'We use a weight decay of 0.0001 and momentum of 0.9' (p.7)
optimizer = optim.SGD(net.parameters(), lr=args.lr,
                      momentum=0.9, weight_decay=0.0001)
# 'We start with a learning rate of 0.1, divide it by 10 at 32k and 48k iterations, and terminate training at 64k iterations' (p.7)
# This code terminates training at the 200 epoch, so the learning rate is divided by 10 at the 100 and 150 epochs to match the same ratio as in the paper
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1)

==> Building Model..

==> Model Summary
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             432
       BatchNorm2d-2           [-1, 16, 32, 32]              32
            Conv2d-3           [-1, 16, 32, 32]           2,304
       BatchNorm2d-4           [-1, 16, 32, 32]              32
            Conv2d-5           [-1, 16, 32, 32]           2,304
       BatchNorm2d-6           [-1, 16, 32, 32]              32
        BasicBlock-7           [-1, 16, 32, 32]               0
            Conv2d-8           [-1, 16, 32, 32]           2,304
       BatchNorm2d-9           [-1, 16, 32, 32]              32
           Conv2d-10           [-1, 16, 32, 32]           2,304
      BatchNorm2d-11           [-1, 16, 32, 32]              32
       BasicBlock-12           [-1, 16, 32, 32]               0
           Conv2d-13           [-1, 16, 32, 32]           2,304

In [6]:
# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        progress_bar(batch_idx, len(trainloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                     % (train_loss/(batch_idx+1), 100.*correct/total, correct, total))


def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar(batch_idx, len(testloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                         % (test_loss/(batch_idx+1), 100.*correct/total, correct, total))

    # Save checkpoint.
    acc = 100.*correct/total
    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

In [7]:
for epoch in range(start_epoch, start_epoch+200):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0
  Step: 705ms | Tot: 31s379ms | Loss: 1.720 | Acc: 36.314% (18157/50000) 782/782 9/782 593/782 
  Step: 9ms | Tot: 996ms | Loss: 1.519 | Acc: 45.030% (4503/10000) 100/100 
Saving..

Epoch: 1
  Step: 43ms | Tot: 30s631ms | Loss: 1.266 | Acc: 54.510% (27255/50000) 782/782 
  Step: 9ms | Tot: 1s13ms | Loss: 1.540 | Acc: 53.930% (5393/10000) 100/100 
Saving..

Epoch: 2
  Step: 41ms | Tot: 30s507ms | Loss: 1.004 | Acc: 64.218% (32109/50000) 782/782 
  Step: 10ms | Tot: 1s31ms | Loss: 1.040 | Acc: 65.770% (6577/10000) 100/100 
Saving..

Epoch: 3
  Step: 41ms | Tot: 30s699ms | Loss: 0.838 | Acc: 70.588% (35294/50000) 782/782 
  Step: 10ms | Tot: 1s46ms | Loss: 1.046 | Acc: 66.720% (6672/10000) 100/100 
Saving..

Epoch: 4
  Step: 41ms | Tot: 30s712ms | Loss: 0.737 | Acc: 74.254% (37127/50000) 782/782 /782 
  Step: 10ms | Tot: 1s56ms | Loss: 0.769 | Acc: 73.750% (7375/10000) 100/100 
Saving..

Epoch: 5
  Step: 41ms | Tot: 30s773ms | Loss: 0.667 | Acc: 76.906% (38453/50000) 782/782 /78

  Step: 41ms | Tot: 30s840ms | Loss: 0.302 | Acc: 89.424% (44712/50000) 782/782 32/782 
  Step: 9ms | Tot: 963ms | Loss: 0.456 | Acc: 85.450% (8545/10000) 100/100 

Epoch: 86
  Step: 41ms | Tot: 30s693ms | Loss: 0.302 | Acc: 89.528% (44764/50000) 782/782 0/782 
  Step: 9ms | Tot: 967ms | Loss: 0.640 | Acc: 80.810% (8081/10000) 100/100 

Epoch: 87
  Step: 42ms | Tot: 30s838ms | Loss: 0.300 | Acc: 89.568% (44784/50000) 782/782 17/78 420/78 612/782 639/78 671/782 
  Step: 9ms | Tot: 988ms | Loss: 0.479 | Acc: 84.850% (8485/10000) 100/100 35/100 

Epoch: 88
  Step: 41ms | Tot: 31s57ms | Loss: 0.300 | Acc: 89.592% (44796/50000) 782/782  /782 577/782 608/782 
  Step: 8ms | Tot: 973ms | Loss: 0.435 | Acc: 85.900% (8590/10000) 100/100 

Epoch: 89
  Step: 41ms | Tot: 30s988ms | Loss: 0.304 | Acc: 89.522% (44761/50000) 782/782 465/78 537/782 
  Step: 9ms | Tot: 971ms | Loss: 0.501 | Acc: 83.620% (8362/10000) 100/100 /100 

Epoch: 90
  Step: 41ms | Tot: 31s221ms | Loss: 0.298 | Acc: 89.592% (4479

  Step: 8ms | Tot: 966ms | Loss: 0.304 | Acc: 92.170% (9217/10000) 100/100 

Epoch: 172
  Step: 41ms | Tot: 30s507ms | Loss: 0.025 | Acc: 99.300% (49650/50000) 782/782 82 626/782 
  Step: 9ms | Tot: 973ms | Loss: 0.307 | Acc: 92.020% (9202/10000) 100/100 

Epoch: 173
  Step: 41ms | Tot: 30s651ms | Loss: 0.023 | Acc: 99.398% (49699/50000) 782/782 1/782 
  Step: 9ms | Tot: 969ms | Loss: 0.307 | Acc: 92.140% (9214/10000) 100/100 

Epoch: 174
  Step: 42ms | Tot: 30s682ms | Loss: 0.024 | Acc: 99.340% (49670/50000) 782/782 
  Step: 9ms | Tot: 965ms | Loss: 0.309 | Acc: 92.080% (9208/10000) 100/100 

Epoch: 175
  Step: 40ms | Tot: 30s638ms | Loss: 0.023 | Acc: 99.328% (49664/50000) 782/782 47/782 744/782 776/782 
  Step: 9ms | Tot: 975ms | Loss: 0.303 | Acc: 92.410% (9241/10000) 100/100 
Saving..

Epoch: 176
  Step: 41ms | Tot: 30s716ms | Loss: 0.024 | Acc: 99.306% (49653/50000) 782/782 1/78 544/782 
  Step: 9ms | Tot: 971ms | Loss: 0.313 | Acc: 92.280% (9228/10000) 100/100 

Epoch: 177
  Ste

In [8]:
print('Accuracy:', round(best_acc, 2))
print('Error:', round(100-best_acc, 2))

Accuracy: 92.41
Error: 7.59
